# Face → depth map (Depth Anything V2)

Before running: **Runtime → Change runtime type → T4 GPU**. Then run the cells top to bottom.

Output files:
- `depth_16bit.png` – smooth 16-bit depth map (white = close, black = far). Use this for the flow field.
- `depth_smooth_16bit.png` – same, lightly blurred, for calmer, rounder lines.
- `depth_preview.png` – 8-bit version just for viewing.

In [ ]:
!pip install -q -U transformers
import torch, numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from google.colab import files
from transformers import pipeline
print("GPU:", torch.cuda.is_available())

## 1. Upload your photo

In [ ]:
up = files.upload()
fname = list(up)[0]
img = ImageOps.exif_transpose(Image.open(fname)).convert("RGB")  # fixes phone rotation
w, h = img.size
print(fname, w, "x", h)
img

## 2. Run the depth model
`Large` gives the best detail around nose and lips (~1.3 GB download). Switch to `Small` if it's slow.

In [ ]:
MODEL = "depth-anything/Depth-Anything-V2-Large-hf"   # or ...-Small-hf
pipe = pipeline("depth-estimation", model=MODEL, device=0 if torch.cuda.is_available() else -1)

result = pipe(img)
pred = result["predicted_depth"]
pred = pred.reshape(1, 1, *pred.shape[-2:]).float()
# upscale the raw prediction to the full photo size (smooth, no 8-bit banding)
depth = torch.nn.functional.interpolate(pred, size=(h, w), mode="bicubic", align_corners=False)[0, 0].cpu().numpy()
depth = (depth - depth.min()) / (depth.max() - depth.min())   # 0..1, 1 = closest
print(depth.shape, depth.min(), depth.max())

## 3. Smooth it (optional but recommended)
Raw depth has tiny bumps (skin texture, hair strands) that make streamlines jittery. Raise `SMOOTH` for softer, more flowing lines; lower it to keep more detail.

In [ ]:
SMOOTH = 1 / 250   # blur radius as a fraction of image size
sigma = max(h, w) * SMOOTH
depth_s = cv2.GaussianBlur(depth.astype(np.float32), (0, 0), sigma)
depth_s = (depth_s - depth_s.min()) / (depth_s.max() - depth_s.min())

fig, ax = plt.subplots(1, 3, figsize=(18, 7))
ax[0].imshow(img); ax[0].set_title("photo")
ax[1].imshow(depth_s, cmap="gray"); ax[1].set_title("depth (smoothed)")
ax[2].contour(depth_s, levels=70, colors="k", linewidths=0.4); ax[2].invert_yaxis()
ax[2].set_aspect("equal"); ax[2].set_title("contour preview – how lines will wrap the face")
for a in ax: a.axis("off")
plt.show()

## 4. Save and download

In [ ]:
cv2.imwrite("depth_16bit.png", (depth * 65535).astype(np.uint16))
cv2.imwrite("depth_smooth_16bit.png", (depth_s * 65535).astype(np.uint16))
cv2.imwrite("depth_preview.png", (depth_s * 255).astype(np.uint8))
for f in ["depth_16bit.png", "depth_smooth_16bit.png", "depth_preview.png"]:
    files.download(f)